# Week 3 · Logistic regression in pictures

Your log on Oct 24: *"I can't picture what is going on in my head... like are we pulling the boundary or something?"* **Yes, you are.** This notebook draws every formula of logistic regression on one small dataset of 9 points, so you can watch what each formula does to the line. The last two sections redo the threshold trade-off and PR-AUC properly. [Week 2 · 05 §4 ↗](../week2/05_logistic_regression_knn.ipynb) gave them one paragraph, which wasn't enough. **Start with section 0**, the 10-minute recap, closed book.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **score $z$**: $z = w^\top x + b$: one number per point, which you can think of as a **height** above the plane of features.
- **boundary**: The line where $z = 0$, so $\hat p = 0.5$.
- **pull $\hat p - y$**: How wrong the model is about one point: positive means the probability is too high, negative means it's too low.
- **threshold $t$**: Predict 1 when $\hat p \ge t$. Changing $t$ doesn't retrain anything.
- **PR-AUC (average precision)**: One number that summarizes precision and recall over all thresholds.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *      # the picture helpers, and the 9 points X_LOGI, Y_LOGI

## 0. Recap first · 10 min, closed book

Answer each one on paper or out loud before opening it. A miss is useful information: write it in your log.

**Quick check:** `a = np.arange(1, 10).reshape(3, 3)`. What are the shapes of `a[:, 0]` and `a[:, 0:1]`?

<details><summary>Answer</summary>

`(3,)` and `(3, 1)`. A number removes its axis; a slice keeps it, even a slice of length 1. You missed this one in Week 2's recap too, so it will keep coming back. [Week 2 · 03 §2, rule 1 ↗](../week2/03_shapes_and_slicing.ipynb)

</details>

**Quick check:** An outcome has probability $\frac{1}{8}$. How surprising is it, in bits?

<details><summary>Answer</summary>

$-\log_2 \frac{1}{8} = \log_2 8 = 3$ bits. [Week 2 · 02 §5, and your note on surprise ↗](../week2/02_map_and_entropy_in_depth.ipynb)

</details>

**Quick check:** Least squares: why is the residual $y - X\hat w$ perpendicular to every column of $X$?

<details><summary>Answer</summary>

If it weren't, moving the prediction $X w$ a little along that column would bring it closer to $y$. At the closest point no direction inside the column space helps any more, which is exactly $X^\top (y - X\hat w) = 0$, the normal equation. [Week 2 · 01 §4, and your projection note ↗](../week2/01_linear_algebra_review.ipynb)

</details>

**Quick check:** Prior $\text{Beta}(3, 3)$ on a coin, and you see 4 heads in 4 flips. What is the MAP estimate?

<details><summary>Answer</summary>

$\hat p = \frac{h + a - 1}{n + a + b - 2} = \frac{4 + 2}{4 + 4} = 0.75$. The MLE would say $1.0$; the prior pulls it toward $0.5$. [Week 2 · 02 §3 ↗](../week2/02_map_and_entropy_in_depth.ipynb)

</details>

**Quick check:** `p = np.array([0.0, 1.0])`, then `np.clip(p, 0.01, 0.99)`, then `print(p)`. What prints?

<details><summary>Answer</summary>

`[0. 1.]`. `np.clip` returns a **new** array and leaves `p` alone; you need `p = np.clip(...)`. [00 §A, point A3 ↗](00_week2_review.ipynb)

</details>

## 1. The picture: points, labels, and a line

**Logistic regression draws one straight line through the plane of features. It says "class 1" on one side and "class 0" on the other. Training means moving that line.**

**The setting.** Each sample has two features $x = (x_1, x_2)$, so it's a **point in the plane**. Its label $y \in \{0, 1\}$ is its colour: blue for 0, orange for 1. These 9 points are used all through this notebook. P5 and P9 sit on the "wrong" side on purpose, because real data overlaps.

|  | P1 | P2 | P3 | P4 | P5 | P6 | P7 | P8 | P9 |
|---|---|---|---|---|---|---|---|---|---|
| $x_1$ | 3 | 2 | 3.5 | 4 | 1 | 1 | 0.5 | 2 | 3 |
| $x_2$ | 2.5 | 3.5 | 3.5 | 1.5 | 1.5 | 0.5 | 2 | 1 | 3 |
| $y$ | 1 | 1 | 1 | 1 | 1 | 0 | 0 | 0 | 0 |

We start from the model in your Week 2 drill, $w = (0.5, 1)$ and $b = -1$ ([05 §3, the logistic-step drill ↗](../week2/05_logistic_regression_knn.ipynb)):

🖼 **Picture.** The starting model is a line. Orange region: the model says 1. Blue region: it says 0. The arrow is $w$.

In [ ]:
fig_setup()

Three things to read off the picture. **The line** is the boundary. **The arrow $w$** stands at a right angle to the line and points into the class-1 side. **The fit is bad**: almost everything is on the orange side, including the blue points P7, P8 and P9, and P6 sits exactly on the line. Training will move this line. The rest of the notebook shows how.

## 2. The score $z$: a height over the plane

**The score $z = w^\top x + b$ gives every point of the plane a number, like a height on a map. The boundary is where the height is 0.**

$$
z = \underbrace{w_1 x_1 + w_2 x_2}_{w^\top x} + \underbrace{b}_{\text{bias}}
$$

Here $x = (x_1, x_2)$ is the point, $w = (w_1, w_2)$ are the weights, one per feature, and $b$ is the bias, which shifts every height by the same amount. With our model, $z = 0.5\,x_1 + 1\,x_2 - 1$.

**What you see vs what happens**

```text
what you see:   z = 0.5·x1 + 1·x2 − 1          one formula

what happens:   every point of the plane gets a height

      x2 = 3 |   2.0   2.5   3.0   3.5   4.0
      x2 = 2 |   1.0   1.5   2.0   2.5   3.0
      x2 = 1 |   0.0   0.5   1.0   1.5   2.0
      x2 = 0 |  −1.0  −0.5   0.0   0.5   1.0
             +-------------------------------
               x1=0  x1=1  x1=2  x1=3  x1=4

    the 0.0 heights, at (0, 1) and (2, 0), lie on one straight line: the boundary
    heights grow fastest toward the upper right: that direction is w
```

**Finding P8's height**

- P8 is $(2, 1)$.
- $w_1 x_1 = 0.5 \times 2 = 1$ and $w_2 x_2 = 1 \times 1 = 1$.
- $z = 1 + 1 - 1 = 1$, so P8 is one unit **uphill** of the boundary, on the class-1 side. That's wrong: P8 is blue.

🖼 **Picture.** Contour lines of $z$, as on a hiking map. They're all parallel to the boundary and evenly spaced, and $w$ points straight uphill.

In [ ]:
fig_score_contours()

**Why $w$ is perpendicular to the boundary.** Take two points $x_a$ and $x_b$ on the line. Both have $z = 0$, so $w^\top x_a + b = w^\top x_b + b$, which gives $w^\top (x_a - x_b) = 0$. The vector $x_a - x_b$ runs **along** the line, and a dot product of 0 means a right angle ([Week 2 · 01 §1 ↗](../week2/01_linear_algebra_review.ipynb)). So $w$ is at $90°$ to every direction along the line.

**Height vs distance.** The signed distance from a point to the line is $\frac{z}{\lVert w \rVert}$. Here $\lVert w \rVert = \sqrt{0.5^2 + 1^2} = 1.118$, so P8 is $\frac{1}{1.118} = 0.894$ away from the line. The sign of $z$ says **which side** a point is on; its size says **how far**.

In [ ]:
X, y = X_LOGI, Y_LOGI                 # (9, 2) and (9,)
w = np.array([0.5, 1.0]); b = -1.0
z = X @ w + b                         # (9, 2) @ (2,) → (9,): one height per point
print(z)                              # → [3.   3.5  4.25 2.5  1.   0.   1.25 1.   3.5 ]
print((z / np.linalg.norm(w)).round(3)[7])   # → 0.894   P8's distance to the line

## 3. The sigmoid: from height to probability

**The sigmoid turns a height into a probability. Far uphill gives nearly 1, far downhill gives nearly 0, and exactly on the line gives 0.5.**

$$
\hat p = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

🖼 **Picture.** Left: P6, P8 and P3 placed on the sigmoid. Right: the same thing done at every point of the plane, so the contour lines are now lines of equal probability.

In [ ]:
fig_sigmoid_and_map()

| Point | height $z$ | $\hat p = \sigma(z)$ | Label | Verdict |
|---|---|---|---|---|
| P6 | 0 | 0.500 | 0 | undecided |
| P8 | 1 | 0.731 | 0 | wrong, fairly sure |
| P3 | 4.25 | 0.986 | 1 | right, very sure |

The right panel is the left panel spread over the plane: each point gets its height, and the sigmoid turns the height into $\hat p$. Lines of equal height become lines of equal probability, so they're still parallel to the boundary. **Logistic regression can only ever draw parallel straight bands like these.** That's its limit, and the reason trees, kernels and neural networks exist.

In [ ]:
p = sigmoid(z)
print(p.round(3))     # → [0.953 0.971 0.986 0.924 0.731 0.5   0.777 0.731 0.971]

## 4. The three knobs: rotate, shift, sharpen

**The direction of $w$ rotates the line, $b$ shifts it, and the length of $w$ sets how sharply $\hat p$ goes from 0 to 1.**

🖼 **Picture.** Each panel turns one knob. Dashed is before, solid is after.

In [ ]:
fig_three_knobs()

| Knob | What changes in the picture | Why |
|---|---|---|
| direction of $w$ | the line rotates | the line is always at $90°$ to $w$ |
| $b$ | the line shifts, staying parallel | the line is where $w^\top x = -b$; changing $-b$ moves it, at distance $\frac{-b}{\lVert w \rVert}$ from the origin |
| length of $w$ (with $b$ scaled too) | same line, sharper band | $z = 0$ is unchanged, but every other height is multiplied, so $\hat p$ reaches 0.1 and 0.9 sooner |

**The band's width, exactly.** $\hat p = 0.9$ when $z = \ln 9 = 2.197$, and $\hat p = 0.1$ when $z = -2.197$. Heights change by $\lVert w \rVert$ per unit of distance, so the band from 0.1 to 0.9 is $\frac{2 \ln 9}{\lVert w \rVert}$ wide. In panel 3, $\lVert w \rVert = 0.944$ gives a width of 4.65; after multiplying by 3 it's 1.55.

**Gradient descent turns all three knobs at once.** Which way each one turns is decided by the points. That's next.

## 5. The loss: the surprise of the true label

**The loss of one point is how surprised the model is by that point's true label: $-\log$ of the probability the model gave to what actually happened.**

> ↩ **Recap** · [Your own note in Week 2 · 02 §5 (Surprise of one outcome) ↗](../week2/02_map_and_entropy_in_depth.ipynb)
>
> You wrote $\text{surprise}(x) = -\log p(x)$: a fair coin landing heads is 1 bit, a 1% event is 6.64 bits, and something certain is 0 bits. The loss below is exactly that, with $\ln$ instead of $\log_2$.

$$
\ell(y, \hat p) = \begin{cases} -\ln \hat p & \text{if } y = 1 \\ -\ln(1 - \hat p) & \text{if } y = 0 \end{cases} \;=\; -\big[\, y \ln \hat p + (1 - y) \ln(1 - \hat p) \,\big]
$$

$\hat p$ is the model's probability of class 1, so the probability it gave to class 0 is $1 - \hat p$. The one-line version on the right uses $y$ as a switch: when $y = 1$ the second term vanishes, and when $y = 0$ the first one does.

🖼 **Picture.** The loss for each label. Right and sure costs almost nothing; wrong and sure costs a lot, and the cost goes to infinity as $\hat p$ approaches the wrong extreme.

In [ ]:
fig_losses()

| Point | Label | $\hat p$ | Probability given to the truth | Loss |
|---|---|---|---|---|
| P3 | 1 | 0.986 | 0.986 | 0.014 |
| P5 | 1 | 0.731 | 0.731 | 0.313 |
| P6 | 0 | 0.500 | 0.500 | 0.693 |
| P8 | 0 | 0.731 | 0.269 | 1.313 |
| P9 | 0 | 0.971 | 0.029 | 3.530 |

P9 alone costs 3.53: the model is 97% sure P9 is orange, and it's blue. The **training loss** is the average over all 9 points, 0.836. Training lowers it.

In [ ]:
loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))    # (9,): one surprise per point
print(loss.round(3))          # → [0.049 0.03  0.014 0.079 0.313 0.693 1.502 1.313 3.53 ]
print(round(loss.mean(), 3))  # → 0.836

## 6. The gradient: every point pulls on the line

**Each point pulls on the boundary with strength $\hat p - y$, in the direction of its own position $x$. The gradient is the average of all the pulls.**

**The formulas, from [Week 2 · 05 §3 ↗](../week2/05_logistic_regression_knn.ipynb).** For one point, the derivative of the loss with respect to its height is $\frac{\partial \ell}{\partial z} = \hat p - y$. Since $z = w^\top x + b$:

$$
\frac{\partial \ell}{\partial w} = \underbrace{(\hat p - y)}_{\text{how wrong}}\; \underbrace{x}_{\text{where the point is}} \qquad\qquad \frac{\partial \ell}{\partial b} = \hat p - y
$$

| Label $y$ | $\hat p$ | Pull $\hat p - y$ | Meaning |
|---|---|---|---|
| 1 | 0.95 | −0.05 | right and sure: barely pulls |
| 1 | 0.20 | −0.80 | wrong and sure: pulls hard, wants its $z$ higher |
| 0 | 0.10 | +0.10 | right and sure: barely pulls |
| 0 | 0.97 | +0.97 | wrong and sure: pulls hard, wants its $z$ lower |

🖼 **Picture.** The starting model again, with each dot sized by how hard it pulls. The three blue points on the orange side (P7, P8, P9) pull the most.

In [ ]:
fig_pulls()

**What one pull does to the line**

The update for one point is $w \leftarrow w - \eta\,(\hat p - y)\,x$. So the step **adds a multiple of $x$ to $w$**, and adding a bit of $x$ to $w$ turns $w$ toward $x$. Because the line is perpendicular to $w$, turning $w$ turns the line.

```text
label 1, p̂ too low      p̂ − y < 0     step = −η(p̂ − y)·x = +(something)·x     w turns TOWARD x
label 0, p̂ too high     p̂ − y > 0     step = −η(p̂ − y)·x = −(something)·x     w turns AWAY from x

either way, the line turns until the point is on its own colour's side
```

🖼 **Picture.** One point, with $b = 0$ so that the line passes through the origin. $x = (1, 2)$ has label 1 but sits exactly on the line of $w = (1, -0.5)$. One step with $\eta = 1$ turns $w$ toward $x$.

In [ ]:
fig_rotation()

**That step, by hand**

- $z = 1 \times 1 + (-0.5) \times 2 = 0$, so $\hat p = 0.5$.
- Pull: $\hat p - y = 0.5 - 1 = -0.5$.
- Step: $-\eta(\hat p - y)\,x = -1 \times (-0.5) \times (1, 2) = (0.5, 1)$.
- New $w = (1, -0.5) + (0.5, 1) = (1.5, 0.5)$. New $z = 1.5 + 0.5 \times 2 = 2.5$, so $\hat p = 0.924$: the point is now well inside the orange side.

🖼 **Picture.** The other direction: $x = (2, 0.5)$ has label 0, but $\hat p = 0.92$. The pull is positive, so the step subtracts a multiple of $x$: it adds $(-1.85, -0.46)$, which turns $w$ away from $x$.

In [ ]:
fig_rotation(w=(1, 1), x=(2, 0.5), y=0)

**And $b$?** Its gradient is the average pull, $\frac{1}{n} \sum_i (\hat p_i - y_i)$. If the model predicts too much class 1 overall (the pulls add up positive), $b$ goes down: every height drops, so the line **shifts** toward the orange side and fewer points are predicted 1.

**In one sentence:** points that are wrong and sure pull hard, points that are right and sure barely pull, and each step moves the line in the direction of the total pull.

## 7. One full step on all 9 points, by hand

> ↩ **Recap** · [Your picture in Week 2 · 05 (logreg_gd_step.png) ↗](../week2/05_logistic_regression_knn.ipynb)
>
> In Week 2 you worked through one point, $x = (2, 1)$ with $y = 0$ (that's P8), from $w = (0.5, 1)$, $b = -1$, and drew the boundary rotating past it. Now all nine points pull at the same time, and the step is their **average**.

Starting model $w = (0.5, 1)$, $b = -1$, learning rate $\eta = 0.5$. One row per point:

| Point | $y$ | $z$ | $\hat p$ | $\hat p - y$ | $(\hat p - y)\,x_1$ | $(\hat p - y)\,x_2$ |
|---|---|---|---|---|---|---|
| P1 | 1 | 3.00 | 0.953 | −0.047 | −0.142 | −0.119 |
| P2 | 1 | 3.50 | 0.971 | −0.029 | −0.059 | −0.103 |
| P3 | 1 | 4.25 | 0.986 | −0.014 | −0.049 | −0.049 |
| P4 | 1 | 2.50 | 0.924 | −0.076 | −0.303 | −0.114 |
| P5 | 1 | 1.00 | 0.731 | −0.269 | −0.269 | −0.403 |
| P6 | 0 | 0.00 | 0.500 | +0.500 | +0.500 | +0.250 |
| P7 | 0 | 1.25 | 0.777 | +0.777 | +0.389 | +1.555 |
| P8 | 0 | 1.00 | 0.731 | +0.731 | +1.462 | +0.731 |
| P9 | 0 | 3.50 | 0.971 | +0.971 | +2.912 | +2.912 |
| **sum** |  |  |  | **+2.543** | **+4.440** | **+4.660** |

- Gradient for $w$: divide the two column sums by $n = 9$: $\nabla_w L = (\frac{4.440}{9}, \frac{4.660}{9}) = (0.4934, 0.5178)$.
- Gradient for $b$: $\frac{2.543}{9} = 0.2826$.
- New $w = (0.5, 1) - 0.5 \times (0.4934, 0.5178) = (0.2533, 0.7411)$.
- New $b = -1 - 0.5 \times 0.2826 = -1.1413$.
- The loss drops from 0.836 to 0.610.

**Who did the pulling?** The biggest entries come from P9, P8 and P7: blue points on the orange side. Their pulls are positive, so they **subtract** from $w$ and $b$. Every height drops, and the line moves up and to the right, toward them. The orange points are already right, so their pulls are small.

In [ ]:
eta = 0.5
e = sigmoid(X @ w + b) - y            # (9,)   one pull per point
grad_w = X.T @ e / len(y)             # (2, 9) @ (9,) → (2,): one number per weight
grad_b = e.mean()                     # ()     one number
print(grad_w.round(4), round(grad_b, 4))          # → [0.4934 0.5178] 0.2826
w1, b1 = w - eta * grad_w, b - eta * grad_b
print(w1.round(4), round(b1, 4))                  # → [0.2533 0.7411] -1.1413

`X.T @ e` is the "sum of each row $x_i$ times its pull $e_i$": exactly the two column sums in the table. That's the same shape pattern as linear regression, $X^\top(\text{error})$.

🖼 **Picture.** Before (dashed) and after (solid) one step. The line moved toward the blue points that were pulling.

In [ ]:
fig_one_step()

## 8. Many steps: the pulls balance out

🖼 **Picture.** The line after 0, 1, 5, 20, 100 and 3000 steps, and the average loss (the step axis is stretched near 0).

In [ ]:
fig_many_steps()

The first steps move the line a lot, because P7, P8 and P9 pull hard. After about 100 steps it barely moves. At step 3000, $w = (0.672, 0.663)$ and $b = -2.610$: roughly the line $x_1 + x_2 = 3.9$, a clean diagonal between the two groups. P5 and P9 stay on the wrong side: **no straight line can fix them**, and that's fine.

**Why it stops: a tug-of-war that balances.** At the final model, the pulls are:

| Point | P1 | P2 | P3 | P4 | P5 | P6 | P7 | P8 | P9 |
|---|---|---|---|---|---|---|---|---|---|
| $\hat p - y$ | −0.256 | −0.258 | −0.113 | −0.255 | −0.720 | +0.167 | +0.279 | +0.354 | +0.802 |

P5 pulls hard one way (it wants the line further down and to the left), P9 pulls hard the other way, and everything else adds up so that

$$
\sum_i (\hat p_i - y_i)\, x_i = (0, 0) \qquad \text{and} \qquad \sum_i (\hat p_i - y_i) = 0
$$

Both gradients are zero, so the step is zero and the line stops. The second equation says $\sum_i \hat p_i = \sum_i y_i$: **at the end, the average predicted probability equals the fraction of class 1**, here $\frac{5}{9}$.

In [ ]:
wt, bt = w.copy(), b
for _ in range(3000):
    e = sigmoid(X @ wt + bt) - y
    wt, bt = wt - eta * X.T @ e / len(y), bt - eta * e.mean()
e = sigmoid(X @ wt + bt) - y
print(wt.round(3), round(bt, 3))              # → [0.672 0.663] -2.61
print(np.allclose(X.T @ e, 0), np.isclose(e.sum(), 0))   # → True True   the pulls cancel
print(round(sigmoid(X @ wt + bt).sum(), 3))   # → 5.0   = number of orange points

**And if no point is on the wrong side?** If a line separates the classes perfectly, every point can always be made *more* sure by making $w$ longer (knob 3). The pulls never cancel, $\lVert w \rVert$ grows forever, and the line keeps getting sharper. That's why scikit-learn's `LogisticRegression` adds an L2 penalty by default (its `C` is $\frac{1}{\lambda}$): the same idea as MAP with a Gaussian prior, i.e. ridge ([Week 2 · 02 §4 ↗](../week2/02_map_and_entropy_in_depth.ipynb)).

## 9. The threshold: sliding the line

**Changing the threshold $t$ slides the decision line parallel to itself. Nothing is retrained: the same $w$ and $b$ are used.**

$$
\hat p \ge t \iff \sigma(z) \ge t \iff z \ge \ln \frac{t}{1 - t}
$$

**Where it comes from.** The sigmoid only ever goes up, so $\sigma(z) \ge t$ is the same as $z \ge \sigma^{-1}(t)$. Solving $t = \frac{1}{1 + e^{-z}}$ for $z$ gives $z = \ln \frac{t}{1 - t}$, the log-odds from [05 §1 ↗](../week2/05_logistic_regression_knn.ipynb). So **a threshold on $\hat p$ is a threshold on the height $z$**, and a fixed height is one of the parallel contour lines from section 2.

| $t$ | line $z = \ln\frac{t}{1-t}$ | Predicted 1 | TP | FP | FN | Precision | Recall |
|---|---|---|---|---|---|---|---|
| 0.25 | −1.099 | 8 points | 5 | 3 | 0 | 0.625 | 1.000 |
| 0.50 | 0 | 5 points | 4 | 1 | 1 | 0.800 | 0.800 |
| 0.85 | +1.735 | 1 point | 1 | 0 | 4 | 1.000 | 0.200 |

🖼 **Picture.** The trained model with three thresholds. A lower $t$ means a lower line, so more points are predicted orange.

In [ ]:
fig_thresholds(w=(0.672, 0.663), b=-2.61, ts=(0.25, 0.5, 0.85))

**The trade-off, read off the picture.** Lowering $t$ moves the line down-left. More points land on the orange side, so you catch more real oranges (**recall up**), but you also sweep in more blues (**false positives up**, so precision usually goes down). Raising $t$ does the opposite.

| Situation | Missing a positive costs... | Pick |
|---|---|---|
| defect detection, disease screening | a lot (a broken part ships, a patient goes home) | a low $t$: high recall |
| spam filter, auto-blocking a payment | little, but a false alarm annoys a real person | a high $t$: high precision |

In [ ]:
for t in [0.25, 0.5, 0.85]:
    pred = (sigmoid(X @ wt + bt) >= t).astype(int)
    tp = ((pred == 1) & (y == 1)).sum(); fp = ((pred == 1) & (y == 0)).sum(); fn = ((pred == 0) & (y == 1)).sum()
    print(t, tp, fp, fn, round(tp / (tp + fp), 3), round(tp / (tp + fn), 3))
# → 0.25 5 3 0 0.625 1.0
#   0.5 4 1 1 0.8 0.8
#   0.85 1 0 4 1.0 0.2

## 10. The precision–recall curve and PR-AUC

**The precision–recall (PR) curve shows precision and recall for every threshold at once. PR-AUC, also called average precision, summarizes it in one number.**

**The setting.** A model scored 10 parts, and 4 of them are really defective. Sort the parts by score, from highest to lowest. Setting the threshold just below the $k$-th score means "flag the top $k$". So each row of the table is one threshold:

| $k$ | Score | Defective? | TP | FP | Precision $\frac{TP}{k}$ | Recall $\frac{TP}{4}$ |
|---|---|---|---|---|---|---|
| 1 | 0.95 | yes | 1 | 0 | 1.000 | 0.25 |
| 2 | 0.85 | yes | 2 | 0 | 1.000 | 0.50 |
| 3 | 0.80 | no | 2 | 1 | 0.667 | 0.50 |
| 4 | 0.70 | yes | 3 | 1 | 0.750 | 0.75 |
| 5 | 0.60 | no | 3 | 2 | 0.600 | 0.75 |
| 6 | 0.45 | no | 3 | 3 | 0.500 | 0.75 |
| 7 | 0.40 | yes | 4 | 3 | 0.571 | 1.00 |
| 8 | 0.30 | no | 4 | 4 | 0.500 | 1.00 |
| 9 | 0.20 | no | 4 | 5 | 0.444 | 1.00 |
| 10 | 0.10 | no | 4 | 6 | 0.400 | 1.00 |

Reading down the table is lowering the threshold. **Recall only goes up**, by $\frac{1}{4}$ each time a real defect is flagged. **Precision wiggles**: it drops on every false alarm (rows 3, 5, 6) and recovers a little on every hit (rows 4, 7).

🖼 **Picture.** Left: the table as a curve; each dot is one row, labelled with its threshold. Right: the ROC curve for the same scores.

In [ ]:
fig_pr_roc([1, 1, 0, 1, 0, 0, 1, 0, 0, 0], [0.95, 0.85, 0.8, 0.7, 0.6, 0.45, 0.4, 0.3, 0.2, 0.1])

**PR-AUC (average precision) is the area under that staircase.** Each time recall steps up, add the width of the step times the precision at that row:

$$
\text{AP} = \sum_k (R_k - R_{k-1})\, P_k
$$

$R_k$ and $P_k$ are the recall and precision in row $k$. Only rows where recall increases contribute; those are the rows with a real defect:

$$
\text{AP} = \underbrace{0.25 \times 1}_{k = 1} + \underbrace{0.25 \times 1}_{k = 2} + \underbrace{0.25 \times 0.75}_{k = 4} + \underbrace{0.25 \times 0.571}_{k = 7} = 0.830
$$

In words: **the average of the precisions at the moments you catch each defect.** A perfect model ranks all 4 defects first and gets 1. A model that guesses randomly has precision about equal to the fraction of positives everywhere, here 0.4, so its AP is about **0.4**. For ROC-AUC, random guessing always gives **0.5**, whatever the class balance.

**Why PR-AUC on imbalanced data.** 1000 parts, 10 defective. The model flags 48: 8 real defects and 40 false alarms ($TP = 8$, $FP = 40$, $FN = 2$, $TN = 950$).

| Metric | Calculation | Value | Impression |
|---|---|---|---|
| false positive rate (ROC's x-axis) | $\frac{40}{40 + 950}$ | 0.040 | looks excellent |
| recall | $\frac{8}{10}$ | 0.800 | good |
| precision | $\frac{8}{48}$ | 0.167 | 5 out of 6 alarms are false |

ROC divides the false alarms by the huge number of negatives (990), so they almost disappear. Precision divides them by the number of alarms, so you see them. **When positives are rare, which is the usual case in anomaly detection, look at PR-AUC.** That's why your KAMP peak detector was judged on it.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score
labels = [1, 1, 0, 1, 0, 0, 1, 0, 0, 0]
scores = [0.95, 0.85, 0.8, 0.7, 0.6, 0.45, 0.4, 0.3, 0.2, 0.1]
print(round(average_precision_score(labels, scores), 4))   # → 0.8304   the AP by hand
print(round(roc_auc_score(labels, scores), 4))             # → 0.8333   20 of the 24 (defect, fine) pairs ranked right
prec, rec, thr = precision_recall_curve(labels, scores)    # every row of the table (in increasing threshold order)

**Drill (on paper).** Six parts, sorted by score: $0.9$ (defect), $0.8$ (fine), $0.7$ (defect), $0.6$ (defect), $0.4$ (fine), $0.2$ (fine). Make the table of precision and recall for $k = 1, \dots, 6$, then compute AP and ROC-AUC.

<details><summary>Worked answer</summary>

Precision: $1,\ 0.5,\ 0.667,\ 0.75,\ 0.6,\ 0.5$. Recall: $\frac{1}{3},\ \frac{1}{3},\ \frac{2}{3},\ 1,\ 1,\ 1$. Recall rises at $k = 1, 3, 4$, so $\text{AP} = \frac{1}{3}(1 + 0.667 + 0.75) = 0.806$. ROC-AUC: the defect at 0.9 beats all 3 fine parts, 0.7 beats 2, and 0.6 beats 2, so $\frac{7}{9} = 0.778$.

</details>

---
Next: `02_knn_and_numpy_workout.ipynb`. kNN rebuilt from the smallest piece, then the NumPy shape workout.